# 01 · Өгөгдөл цуглуулах (Data Collection)

**Шуугиан** төслийн эхний алхам: дэлхийн анхаарлыг хэмжих өгөгдлөө цуглуулна.

| Юу | Тайлбар |
|---|---|
| Эх сурвалж | Wikimedia Pageviews API — Wikipedia-ийн өдөр тутмын хандалт |
| Хугацаа | 2021-01-01 → өчигдөр |
| Сэдэв | 10 ангиллын ~150 **сонгосон** сэдэв + ~150 өнгөрсөн **вирал** сэдэв |
| Нэмэлт | Англи Wikipedia-ийн **нийт** өдөр тутмын хандалт |

> 💡 **Яагаад Wikipedia вэ?** Хүн ямар нэг зүйлийг сонирхохдоо түүнийг хайж уншдаг.
> Wikipedia-ийн хандалт бол "дэлхий юуг сонирхож байна" гэдгийн үнэгүй, өдөр тутмын,
> 10 жилийн түүхтэй хэмжүүр. Google Trends-ийн албан ёсны API одоогоор хаалттай.

## 1. Тохиргоо

`REFRESH = False` үед git-д хадгалагдсан бэлэн өгөгдлийг уншина (хурдан).
`True` болговол Wikipedia-аас **шинээр** татна (~40–60 мин, rate limit-ээс болж).

In [1]:
import sys
from pathlib import Path

import pandas as pd
import plotly.express as px
import plotly.io as pio

pio.templates.default = "plotly_dark"   # сайт маань dark горимтой тул графикууд ч мөн адил

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))           # src/ доторх кодыг import хийх боломжтой болгоно
DATA = ROOT / "data"

REFRESH = False
if REFRESH:
    from src import fetch_data
    fetch_data.main()

## 2. Өгөгдөл хэрхэн татагдсан бэ?

Бүх код `src/wiki.py` болон `src/fetch_data.py` файлд байгаа. Гол санаанууд:

1. **`agent=user`** — зөвхөн **хүний** хандалтыг авна. Wikipedia ботуудыг өөрөө ялгаж хасдаг.
2. **Rate limit** — API цагт хязгаартай хүсэлт зөвшөөрдөг. Тиймээс хүсэлт бүрийн хооронд
   хүлээж, `429` алдаа ирвэл заасан хугацаагаар хүлээгээд дахин оролддог.
3. **Cache** — татсан хариу бүрийг `data/raw/cache/` дотор хадгална. Дахин ажиллуулахад дискнээс уншина.
4. **Вирал сэдэв сонгох** — 2021 оноос хойш **сар бүрийн топ-1000** жагсаалтыг татаад,
   зөвхөн **1–2 сар** гарч ирсэн (гэнэт шуугиад алга болсон) сэдвүүдийг нэр дэвшүүлнэ.
   Дараа нь оргил өдөр нь медианаасаа **8+ дахин** их байгааг шалгаж баталгаажуулна.
5. **Нэр солигдсон нийтлэл** — жишээ нь "J. D. Vance" 2024 онд "JD Vance" болсон. Хуучин нэрийн
   хандалт шинэ нэрэнд тооцогддоггүй тул хуучин нэрсийг (`data/aliases.csv`) тусад нь татсан.
   Тэдгээрийг 02 notebook-д нэгтгэнэ.
6. **Тохиромжгүй сэдэв** — олон нийтийн сайтад гаргах тул насанд хүрэгчдийн контентыг
   (`data/exclude.csv`) гараар хассан.

> 💡 **Яагаад вирал сэдэв нэмэх вэ?** Сонгосон 150 сэдэв ихэвчлэн тогтвортой. Хэрэв model зөвхөн
> тэднээс сурвал "гэнэт шуугисан сэдэв хэрхэн унтардаг"-ийг хэзээ ч харахгүй. Вэбийн
> **🔥 Шуугиан** хуудас яг ийм сэдвүүдийг таамаглах тул model-д жишээ хэрэгтэй.

## 3. Өгөгдлөө ачаалах

In [2]:
pv = pd.read_parquet(DATA / "processed" / "pageviews.parquet")      # сэдэв бүрийн өдөр тутмын хандалт
total = pd.read_parquet(DATA / "processed" / "total_views.parquet")  # Wikipedia-ийн нийт хандалт
articles = pd.read_csv(DATA / "articles.csv")
viral = pd.read_csv(DATA / "viral_articles.csv")

print(f"Мөр: {len(pv):,}")
print(f"Сэдэв: {pv['article'].nunique()}  (сонгосон {articles.shape[0]}, вирал {viral.shape[0]})")
print(f"Хугацаа: {pv['date'].min().date()} → {pv['date'].max().date()}")
pv.head()

Мөр: 586,420
Сэдэв: 301  (сонгосон 151, вирал 150)
Хугацаа: 2021-01-01 → 2026-11-06


,date,article,category,views
0,2021-01-01,1980_United_States_presidential_election,viral,3991
1,2021-01-02,1980_United_States_presidential_election,viral,4040
2,2021-01-03,1980_United_States_presidential_election,viral,4376
3,2021-01-04,1980_United_States_presidential_election,viral,7675
4,2021-01-05,1980_United_States_presidential_election,viral,6110


In [3]:
# Багана бүрийн төрөл, санах ойн хэмжээ
pv.info(memory_usage="deep")

<class 'pandas.DataFrame'>
RangeIndex: 586420 entries, 0 to 586419
Data columns (total 4 columns):
 #   Column    Non-Null Count   Dtype         
---  ------    --------------   -----         
 0   date      586420 non-null  datetime64[us]
 1   article   586420 non-null  str           
 2   category  586420 non-null  str           
 3   views     586420 non-null  int64         
dtypes: datetime64[us](1), int64(1), str(2)
memory usage: 29.9 MB


## 4. Ангилал бүрт хэдэн сэдэв байна?

In [4]:
cat_names = {
    "politics": "🏛️ Улс төр", "film_tv": "🎬 Кино/цуврал", "true_crime": "🕯️ True crime",
    "tech_ai": "🤖 AI/технологи", "sports": "⚽ Спорт", "music": "🎵 Хөгжим",
    "games_anime": "🎮 Тоглоом/anime", "celebs": "💅 Алдартнууд", "society": "👥 Нийгэм",
    "mongolia": "🇲🇳 Монгол", "viral": "🔥 Вирал",
}
pv["category_mn"] = pv["category"].map(cat_names)
pv.groupby("category_mn")["article"].nunique().sort_values(ascending=False).to_frame("сэдвийн тоо")

,сэдвийн тоо
category_mn,
🔥 Вирал,150
🏛️ Улс төр,16
⚽ Спорт,15
🎬 Кино/цуврал,15
🇲🇳 Монгол,15
🎵 Хөгжим,15
🎮 Тоглоом/anime,15
👥 Нийгэм,15
💅 Алдартнууд,15


## 5. Өгөгдлийн чанарын анхны шалгалт

Хугацаа 2021-01-01-ээс эхэлдэг тул сэдэв бүр ~2,100 өдөртэй байх ёстой. Цөөн өдөртэй бол:
- нийтлэл **хожим үүссэн** (жишээ нь DeepSeek, 2025 оны шинэ кино), эсвэл
- нийтлэлийн **нэр өөрчлөгдсөн** — хуучин нэрийн хандалт шинэ нэрэнд тооцогддоггүй!

Энэ асуудлыг дараагийн notebook-д (`02_cleaning_eda`) шийднэ. Одоохондоо зөвхөн илрүүлье.

In [5]:
expected_days = (pv["date"].max() - pv["date"].min()).days + 1
days = pv.groupby(["article", "category"]).size().rename("өдөр").reset_index()
days["дутуу %"] = (100 * (1 - days["өдөр"] / expected_days)).round(1)
print(f"Хүлээгдэж буй өдөр: {expected_days}")
days.sort_values("өдөр").head(15)

Хүлээгдэж буй өдөр: 2136


,article,category,өдөр,дутуу %
226,Prominent_individuals_mentioned_in_the_Epstein...,viral,277,87.0
127,India_at_the_2026_Commonwealth_Games,viral,399,81.3
177,Love_of_Your_Life_(film),viral,402,81.2
65,Coachella_2026,viral,417,80.5
173,Lokah,viral,443,79.3
294,War_of_the_Worlds_(2025_film),viral,471,77.9
120,Iceman_(Drake_album),viral,490,77.1
293,WWE_Evolution_(2025),viral,531,75.1
192,Money_in_the_Bank_(2026),viral,533,75.0
12,2026_Ballon_d'Or,viral,542,74.6


## 6. Wikipedia-ийн нийт хандалт буурч байна уу?

Wikimedia сангийн мэдээлснээр Wikipedia-ийн (бүх хэл) хүний уншилт 2025 оны сүүлийн хэдэн сард
2024 оны мөн саруудтай харьцуулахад ~8% буурсан ([Diff, 2025-10-17](https://diff.wikimedia.org/2025/10/17/new-user-trends-on-wikipedia/)).
Сан үүнийг генератив AI болон сошиал медиагийн нөлөө гэж тайлбарласан.
Үүнийг өөрсдийн өгөгдлөөр шалгая. **7 хоногийн гулсах дундаж** нь өдөр бүрийн "шуугиан"-ыг тэгшилж,
ерөнхий чиг хандлагыг харуулна.

In [6]:
t = total.copy()
t["7 хоногийн дундаж"] = t["total_views"].rolling(7).mean()
fig = px.line(t, x="date", y="7 хоногийн дундаж",
              title="Англи Wikipedia-ийн өдөр тутмын нийт хүний хандалт")
fig.update_layout(yaxis_title="хандалт / өдөр", xaxis_title=None)
fig.show()

> 💡 **Яагаад энэ чухал вэ?** Хэрэв бид шууд хандалтын **тоог** таамаглавал model
> "Wikipedia бүхэлдээ буурч байна" гэдгийг "сэдэв сонирхол алдаж байна" гэж андуурна.
> Тиймээс **анхаарлын хувь**-ийг ашиглана:
>
> $$\text{анхаарал} = \frac{\text{сэдвийн хандалт}}{\text{тухайн өдрийн нийт хандалт}}$$
>
> Энэ нь "тэр өдөр дэлхий Wikipedia-д зарцуулсан анхаарлын хэдэн хувийг энэ сэдэв авсан бэ" гэсэн утгатай.

## 7. Жишээ: Mongolia — тоо vs анхаарлын хувь

In [7]:
m = pv[pv["article"] == "Mongolia"].merge(total, on="date")
m["анхаарал (сая-д)"] = 1e6 * m["views"] / m["total_views"]   # 1 сая хандалт тутамд хэд нь Mongolia байв
m = m.set_index("date")[["views", "анхаарал (сая-д)"]].rolling(7).mean().reset_index()

fig = px.line(m, x="date", y=["views", "анхаарал (сая-д)"], facet_row="variable",
              title="Mongolia: хандалтын тоо (дээр) ба анхаарлын хувь (доор), 7 хоногийн дундаж")
fig.update_yaxes(matches=None, title=None)
fig.for_each_annotation(lambda a: a.update(text=a.text.split("=")[-1]))
fig.show()

## 8. Жишээ: вирал сэдэв хэрхэн "шуугиад" унтардаг вэ?

Хамгийн огцом өссөн 4 вирал сэдвийг харъя. Ихэнх нь хэдхэн хоногт оргилдоо хүрээд,
дараа нь **экспоненциал** маягаар (эхэндээ хурдан, дараа нь удаан) буурдаг.
Яг энэ "унтрах хэлбэр"-ийг model сурах ёстой.

In [8]:
top4 = viral.sort_values("ratio", ascending=False)["article"].head(4).tolist()
v = pv[pv["article"].isin(top4)].copy()

# Сэдэв бүрийн оргил өдрийг "0-р өдөр" болгож, ±30 хоногийг харьцуулна
peak = v.loc[v.groupby("article")["views"].idxmax(), ["article", "date"]].rename(columns={"date": "peak"})
v = v.merge(peak, on="article")
v["оргилоос хойш (өдөр)"] = (v["date"] - v["peak"]).dt.days
v = v[v["оргилоос хойш (өдөр)"].between(-10, 30)]

fig = px.line(v, x="оргилоос хойш (өдөр)", y="views", color="article", log_y=True,
              title="Вирал сэдвүүд: оргилын өмнөх ба дараах хандалт (log хуваарь)")
fig.show()

## 9. Ангилал бүрийн "ердийн" анхаарал

Медиан (дунд утга) ашиглаж байгаа нь цөөн хэдэн огцом өсөлт (outlier) дүгнэлтийг гажуудуулахгүй байх зорилготой.

In [9]:
s = pv.merge(total, on="date")
s["анхаарал (сая-д)"] = 1e6 * s["views"] / s["total_views"]
by_cat = (s.groupby(["category_mn", "article"])["анхаарал (сая-д)"].median()
           .groupby("category_mn").median().sort_values().reset_index())
fig = px.bar(by_cat, x="анхаарал (сая-д)", y="category_mn", orientation="h",
             title="Ангиллын ердийн сэдэв: 1 сая хандалт тутамд хэдэн удаа уншигддаг вэ (медиан)")
fig.update_layout(yaxis_title=None)
fig.show()

## ✅ Дүгнэлт ба дараагийн алхам

- ~300 сэдвийн 2021 оноос хойшхи өдөр тутмын **хүний** хандалтыг цуглууллаа.
- Wikipedia-ийн нийт хандалт өөрчлөгддөг тул **анхаарлын хувь**-ийг target болгоно.
- Зарим сэдэв цөөн өдөртэй — **нэр солигдсон эсвэл хожим үүссэн** эсэхийг шалгах хэрэгтэй.

**Дараагийн notebook (`02_cleaning_eda`)**: дутуу өдрүүдийг нөхөх, нэр солигдсон сэдвүүдийг
илрүүлэх, ботын хэвийн бус өсөлтийг (outlier) олох, долоо хоног болон улирлын хэв маягийг судлах.